# Árboles de Recursión - Análisis y Visualización

Este notebook explora la generación y visualización de árboles de recursión para algoritmos recursivos.

## Configuración Inicial

In [ ]:
import sys
sys.path.append('../..')

from app.core.parser.pseudocode_parser import PseudocodeParser
from app.core.analyzer.recurrence.recurrence_builder import RecurrenceBuilder
from app.core.visualization.recursion_tree_generator import RecursionTreeGenerator
from app.core.visualization.diagram_renderer import DiagramRenderer
from pathlib import Path

# Configurar directorio de salida
output_dir = Path("../../data/exports/notebooks/recursion_trees")
output_dir.mkdir(parents=True, exist_ok=True)

print("Módulos importados correctamente")

## 1. Introducción a los Árboles de Recursión

Los árboles de recursión son representaciones visuales que muestran cómo se expande una llamada recursiva. Cada nodo representa una llamada a la función, y las aristas conectan llamadas padre con sus llamadas hijas.

### Componentes de un Árbol de Recursión

- **Nodo Raíz**: La llamada inicial al algoritmo
- **Nodos Internos**: Llamadas recursivas intermedias
- **Nodos Hoja**: Casos base que terminan la recursión
- **Aristas**: Relaciones de llamada entre funciones

## 2. Ejemplo Básico: Fibonacci

In [ ]:
# Algoritmo Fibonacci recursivo
fibonacci_code = """
ALGORITHM Fibonacci(n)
    IF n <= 1 THEN
        RETURN n
    ENDIF
    
    RETURN Fibonacci(n-1) + Fibonacci(n-2)
END
"""

# Parsear el código
parser = PseudocodeParser()
ast = parser.parse(fibonacci_code)

print("AST generado:")
print(ast)

### Construir Ecuación de Recurrencia

In [ ]:
# Extraer ecuación de recurrencia
recurrence_builder = RecurrenceBuilder()
recurrence = recurrence_builder.build_recurrence(ast)

print("\nEcuación de recurrencia:")
print(f"T(n) = {recurrence}")
print(f"\nCasos base:")
print("  T(0) = O(1)")
print("  T(1) = O(1)")

### Generar Árbol de Recursión

In [ ]:
# Generar árbol para Fibonacci(5)
tree_generator = RecursionTreeGenerator()

tree = tree_generator.generate_tree(
    recurrence=recurrence,
    base_cases={'n <= 1': 'O(1)'},
    options={
        'max_depth': 10,
        'show_cost': True,
        'show_level': True
    }
)

print(f"\n Árbol generado con {tree.node_count} nodos")
print(f"Profundidad: {tree.depth}")

### Visualizar Árbol

In [ ]:
# Renderizar en diferentes formatos
renderer = DiagramRenderer()

# SVG
svg_output = renderer.render(tree, 'svg', {'title': 'Fibonacci(5) - Árbol de Recursión'})
svg_path = output_dir / "fibonacci_tree.svg"
svg_path.write_bytes(svg_output)
print(f" SVG guardado en: {svg_path}")

# DOT (Graphviz)
dot_output = tree.to_dot()
dot_path = output_dir / "fibonacci_tree.dot"
dot_path.write_text(dot_output)
print(f" DOT guardado en: {dot_path}")

# Mermaid
mermaid_output = tree.to_mermaid()
mermaid_path = output_dir / "fibonacci_tree.mmd"
mermaid_path.write_text(mermaid_output)
print(f" Mermaid guardado en: {mermaid_path}")

### Análisis de Complejidad desde el Árbol

In [ ]:
# Analizar el árbol
analysis = tree_generator.analyze_tree(tree)

print("\n Análisis del Árbol:")
print(f"  - Total de nodos: {analysis['total_nodes']}")
print(f"  - Nodos por nivel: {analysis['nodes_per_level']}")
print(f"  - Trabajo por nivel: {analysis['work_per_level']}")
print(f"  - Complejidad total: {analysis['total_complexity']}")

## 3. Ejemplo Medio: QuickSort

In [ ]:
quicksort_code = """
ALGORITHM QuickSort(arr, low, high)
    IF low < high THEN
        pivot_index = Partition(arr, low, high)
        QuickSort(arr, low, pivot_index - 1)
        QuickSort(arr, pivot_index + 1, high)
    ENDIF
END
"""

# Parsear y generar árbol
ast_quick = parser.parse(quicksort_code)
recurrence_quick = recurrence_builder.build_recurrence(ast_quick)

print("Ecuación de recurrencia QuickSort:")
print(f"T(n) = {recurrence_quick}")

### Comparar Mejor y Peor Caso

In [ ]:
# Mejor caso (pivote siempre en el medio)
tree_best = tree_generator.generate_tree(
    recurrence=recurrence_quick,
    base_cases={'n <= 1': 'O(1)'},
    options={
        'case': 'best',
        'n': 16,
        'balanced': True
    }
)

# Peor caso (pivote siempre en el extremo)
tree_worst = tree_generator.generate_tree(
    recurrence=recurrence_quick,
    base_cases={'n <= 1': 'O(1)'},
    options={
        'case': 'worst',
        'n': 16,
        'balanced': False
    }
)

print(f"\n Comparación QuickSort:")
print(f"  Mejor caso: {tree_best.depth} niveles, {tree_best.node_count} nodos")
print(f"  Peor caso: {tree_worst.depth} niveles, {tree_worst.node_count} nodos")

## 4. Ejemplo Avanzado: Merge Sort

In [ ]:
mergesort_code = """
ALGORITHM MergeSort(arr, left, right)
    IF left < right THEN
        mid = (left + right) / 2
        MergeSort(arr, left, mid)
        MergeSort(arr, mid + 1, right)
        Merge(arr, left, mid, right)
    ENDIF
END
"""

ast_merge = parser.parse(mergesort_code)
recurrence_merge = recurrence_builder.build_recurrence(ast_merge)

tree_merge = tree_generator.generate_tree(
    recurrence=recurrence_merge,
    base_cases={'n <= 1': 'O(1)'},
    options={
        'n': 16,
        'show_work': True,
        'annotate_merges': True
    }
)

print(f"\n MergeSort tree: {tree_merge.node_count} nodos, {tree_merge.depth} niveles")

### Visualización Comparativa

In [ ]:
import matplotlib.pyplot as plt

# Comparar profundidad de árboles
algorithms = ['Fibonacci(10)', 'QuickSort(16) Best', 'QuickSort(16) Worst', 'MergeSort(16)']
depths = [10, 4, 16, 4]
node_counts = [177, 31, 136, 31]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Profundidad
ax1.bar(algorithms, depths, color=['#FF6B6B', '#4ECDC4', '#FF8C42', '#45B7D1'])
ax1.set_ylabel('Profundidad del Árbol')
ax1.set_title('Profundidad de Árboles de Recursión')
ax1.tick_params(axis='x', rotation=45)

# Número de nodos
ax2.bar(algorithms, node_counts, color=['#FF6B6B', '#4ECDC4', '#FF8C42', '#45B7D1'])
ax2.set_ylabel('Número de Nodos')
ax2.set_title('Tamaño de Árboles de Recursión')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(output_dir / 'tree_comparison.png', dpi=150, bbox_inches='tight')
print(f" Comparación guardada")

## 5. Optimización de Árboles Grandes

In [ ]:
from app.core.visualization.optimizer import VisualizationOptimizer, OptimizationConfig, OptimizationLevel

# Generar árbol grande
large_tree = tree_generator.generate_tree(
    recurrence="T(n) = 2*T(n-1) + O(1)",
    base_cases={'n <= 0': 'O(1)'},
    options={'n': 20}
)

print(f"\nÁrbol grande: {large_tree.node_count} nodos")

# Optimizar
optimizer = VisualizationOptimizer(
    OptimizationConfig(level=OptimizationLevel.AGGRESSIVE)
)

optimized_tree = optimizer.optimize_recursion_tree(
    {'root': large_tree.root},
    max_depth=10
)

print(f"Árbol optimizado: {optimized_tree['nodes_count']} nodos")
print(f"Reducción: {(1 - optimized_tree['nodes_count'] / large_tree.node_count) * 100:.1f}%")

## 6. Ejercicios Prácticos

### Ejercicio 1: Torres de Hanoi

In [ ]:
hanoi_code = """
ALGORITHM Hanoi(n, source, destination, auxiliary)
    IF n = 1 THEN
        MOVE disk FROM source TO destination
        RETURN
    ENDIF
    
    Hanoi(n-1, source, auxiliary, destination)
    MOVE disk FROM source TO destination
    Hanoi(n-1, auxiliary, destination, source)
END
"""

# TODO: Parsear, generar árbol y visualizar
# Tu código aquí...

### Ejercicio 2: Análisis de Complejidad

In [ ]:
# Para cada algoritmo, determina:
# 1. La ecuación de recurrencia
# 2. La altura del árbol
# 3. El trabajo total
# 4. La complejidad asintótica

algorithms_exercise = [
    "T(n) = T(n-1) + O(1)",           # Factorial
    "T(n) = T(n/2) + O(1)",           # Binary Search
    "T(n) = 2*T(n/2) + O(n)",         # Merge Sort
    "T(n) = 3*T(n/4) + O(n²)",        # Algoritmo personalizado
]

# Analiza cada uno...

## 7. Conclusiones

### Patrones Observados

1. **Árboles Balanceados**: Algoritmos divide y conquista con particiones uniformes
2. **Árboles Desbalanceados**: Algoritmos con particiones no uniformes
3. **Árboles Exponenciales**: Recursión múltiple sin memoization (Fibonacci)

### Relación con Complejidad

- **Altura del árbol**: Relacionada con el tiempo de ejecución
- **Número de nodos**: Número total de llamadas recursivas
- **Trabajo por nivel**: Determina la complejidad final

### Próximos Pasos

- Explorar memoization y programación dinámica
- Visualizar grafos de ejecución
- Analizar complejidad espacial